# Kaggle Competition
Due Nov 11

Team 27: Rachel Liu, Henry Lam, Yang Xiao

The PR should include
* implementation,
* experimental code,
* README,
* dependencies, and
* instructions for generating your Kaggle predictions.

Your PR should preserve meaningful development history, including experiments, debugging, and changes to your approach

## Option 1: Neural Net
0. preprocess
1. pick a **model class**, 
2. an **estimation principle**
3. turn it into a **loss**,
4. **optimize** it, 
5. then **predict and evaluate** 

In [1]:
%matplotlib inline
import re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

import torch
from torch import nn

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, log_loss, roc_auc_score
from scipy.sparse import hstack

SEED = 27   # team number, so every run uses the same split and weight initialization
np.random.seed(SEED)
torch.manual_seed(SEED)

### Step 0: Preprocess Data
Make the raw data (attributes) usable via featurization, PCA.

#### (a) Load the data

In [2]:
train_df = pd.read_csv("data/train.csv")   # id, text, label (1 = AI, 0 = human)
test_df = pd.read_csv("data/test.csv")     # id, text (no labels, this is what we submit predictions for)

print("train.csv shape:", train_df.shape)
print("test.csv  shape:", test_df.shape)
print("fraction labeled AI:", train_df["label"].mean())   # 0.5 means the classes are balanced
train_df.head()

train.csv shape: (10000, 3)
test.csv  shape: (3000, 2)
fraction labeled AI: 0.5


,id,text,label
0,1,Cipriani S.A. is an Italian hotel and leisure ...,0
1,2,My Grandpa passed away 3 years ago from cancer...,0
2,3,"""Petit Beurre -- Butter Cookies\n\n1 cup (2 st...",1
3,4,"In an unprecedented move, a dedicated gamer ha...",1
4,5,Chile's copper industry has registered record ...,0


#### (b) Split the training data
1. Fit on the training set, 
2. use the held-out development set to compare models,
3. pick hyperparameters, and
4. decide when to stop training.

Since the test set has no labels the held-out dev set will be used to compare models.

In [9]:
train_set, dev_set = train_test_split(
    train_df, test_size=0.2, random_state=SEED, stratify=train_df["label"]
)

y_train = train_set["label"].values
y_dev = dev_set["label"].values

print("training set size:   ", len(train_set), f"({y_train.mean()*100}% AI)")
print("development set size:", len(dev_set), f"({y_dev.mean()*100}% AI)")

training set size:    8000 (50.0% AI)
development set size: 2000 (50.0% AI)


#### (c) Clean the text
We keep punctuation and capitalization because *how* the text is written likely plays a large part in whether the text is human versus AI. I'm thinking m-dashes and "you're absolutely right" for AI, and typos for humans.  

In [10]:
def clean_text(text):
    # Collapse runs of spaces/newlines into a single space
    return re.sub(r"\s+", " ", text).strip()

train_text = train_set["text"].map(clean_text)
dev_text = dev_set["text"].map(clean_text)

print(train_text.iloc[0][:400])

An $N$-level quantum system is coupled to a bosonic heat reservoir at positive temperature. We analyze the system-reservoir dynamics in the following regime: The strength $\lambda$ of the system-reservoir coupling is fixed and small, but larger than the spacing $\sigma$ of system energy levels. For vanishing $\sigma$ there is a manifold of invariant system-reservoir states and for $\sigma>0$ the o


#### (d) Featurize: TF-IDF on words and characters
Instead of the 0/1 binary that we've use in the past for bag-of-words, we're now replacing that with weights through TF-IDF. 
Based on research, it seems like best practice to include both a 
* word vectorizer to look at **what** is said (vocabulary, common phrases), and also a 
* character vectorizer to look at **how** it is said (punctuation habits, spelling, word endings).

As always, vectorizers are fit on the training set only, then applied to dev with the same vocabulary.

In [11]:
word_vectorizer = TfidfVectorizer(
    lowercase=True, ngram_range=(1, 2), min_df=3, max_features=20000, sublinear_tf=True
)
char_vectorizer = TfidfVectorizer(
    lowercase=False, analyzer="char_wb", ngram_range=(3, 5), min_df=5, max_features=30000, sublinear_tf=True
)

# hstack adds the two sparse matrices side by side: d = (#word features) + (#char features)
X_train = hstack([word_vectorizer.fit_transform(train_text),
                  char_vectorizer.fit_transform(train_text)]).tocsr()
X_dev = hstack([word_vectorizer.transform(dev_text),
                char_vectorizer.transform(dev_text)]).tocsr()

print("train matrix:", X_train.shape, "   dev matrix:", X_dev.shape)

train matrix: (8000, 50000)    dev matrix: (2000, 50000)


#### (e) Wrap the data for PyTorch
The matrices are sparse which means they are mostly zeros. We only need to turn one **minibatch** of rows into a dense tensor at a time because converting the entire one would take a lot of memory. 

In [12]:
class SparseTextDataset(torch.utils.data.Dataset):
    def __init__(self, X, y=None):
        self.X = X   # scipy sparse matrix, shape (n, d)
        self.y = y   # numpy array of 0/1 labels, or None for the test set

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return i   # just hand back row indices; collate() builds the batch

    def collate(self, idx):
        # Slice the rows for this batch and convert only those to a dense float tensor.
        xb = torch.tensor(self.X[idx].toarray(), dtype=torch.float32)
        if self.y is None:
            return xb
        yb = torch.tensor(self.y[idx], dtype=torch.float32)
        return xb, yb

def make_loader(X, y=None, batch_size=64, shuffle=False):
    ds = SparseTextDataset(X, y)
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=ds.collate)

train_loader = make_loader(X_train, y_train, shuffle=True)   # reshuffle every epoch for SGD
dev_loader = make_loader(X_dev, y_dev)

xb, yb = next(iter(train_loader))
print("one batch:", xb.shape, yb.shape)

one batch: torch.Size([64, 50000]) torch.Size([64])


### Step 1: Parametrize
The model class is a **multilayer perceptron** (MLP) where each layer computes the **activation** of the weighted inputs plus bias, feeding forward per layer. 

In [14]:
# this defines a simple feedforward neural network with dropout and a choice of activation function.
class MLP(nn.Module):
    def __init__(self, sizes, activation="relu", dropout=0.5):
        super().__init__()
        self.sizes = sizes
        # one Linear layer per pair of consecutive sizes, W^[k] has shape (sizes[k], sizes[k-1])
        self.layers = nn.ModuleList([nn.Linear(m, n) for m, n in zip(sizes[:-1], sizes[1:])])
        self.g = nn.ReLU() if activation == "relu" else nn.Sigmoid()
        self.dropout = nn.Dropout(dropout)   # only active in model.train() mode

    def forward(self, x):
        for layer in self.layers[:-1]:
            x = self.dropout(self.g(layer(x)))   # A^[k] = g(W^[k] A^[k-1] + b^[k])
        # last layer: no activation, one logit per example; squeeze(-1) turns (batch, 1) into (batch,)
        return self.layers[-1](x).squeeze(-1)

SIZES = [X_train.shape[1], 256, 64, 1]
model = MLP(SIZES)
print(model)
print("number of parameters:", sum(p.numel() for p in model.parameters()))

MLP(
  (layers): ModuleList(
    (0): Linear(in_features=50000, out_features=256, bias=True)
    (1): Linear(in_features=256, out_features=64, bias=True)
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
  (g): ReLU()
  (dropout): Dropout(p=0.5, inplace=False)
)
number of parameters: 12816769


### Step 2: Choose Estimation Principle
We will use **Maximum likelihood / logistic regression**. We model each label as a
Bernoulli RV whose probability comes from the network and choose theta to make observed training labels as likely as possible which is the process of .

We also add **L2 penalty** to reduce overfitting

### Step 3: Define a Loss/Objective Function
Maximizing the log-likelihood = minimizing its negative, which is the **binary cross-entropy** loss. 
* Pass raw model logits and 0/1 labels; this loss applies sigmoid internally and
* computes the mean binary cross-entropy for the batch.

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()

BCEWithLogitsLoss:       0.6946050524711609
-mean log-likelihood:    0.6946050524711609


### Step 4: Optimize
#### (a) Backpropagation via PyTorch
PyTorch's `loss.backward()` derives the gradient with cross entropy and sigmoid output. It flows backwards by applying the weighted deltas to the layers below it. 